### All scenarios

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
import matplotlib as mpl
import matplotlib.pyplot as plt
# Keep text editable in Adobe Illustrator SVG/PDF outputs
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

from pathlib import Path

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams.update({'font.size': 20})

# ================== 0. Paths ==================
BASE_PATH = Path.cwd().parent.parent

RAW = BASE_PATH / 'Data' / 'raw'
TEMP = BASE_PATH / 'Data' / 'temp'
USE = BASE_PATH / 'Data' / 'use'
FIGURES = BASE_PATH / 'Results' / 'Figures'
TABLES = BASE_PATH / 'Results' / 'Tables'

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

EXCEL_PATH  = USE / 'dc_life_extension_retirement_age_25_36_40_50.xlsx'
COAL_PATH   = RAW / 'Global-Coal-Plant-Tracker-January-2025-cf.xlsx'
CF_PATH     = TABLES / 'CF_trend.csv'
CF_CAP_PATH = RAW / 'Capacity_factor.xlsx'

OUTPUT_TABLE_DIR = TABLES
OUTPUT_FIGURE_DIR = FIGURES

print(f"BASE_PATH: {BASE_PATH}")
print(f"Life-extension input: {EXCEL_PATH}")
print(f"Coal tracker input: {COAL_PATH}")
print(f"CF trend input: {CF_PATH}")
print(f"Capacity factor input: {CF_CAP_PATH}")


def scenario_file_label(cf_col):
    return (
        cf_col
        .replace('°', 'deg')
        .replace('>', 'gt')
        .replace('+', 'plus')
        .replace('–', '-')
        .replace(' ', '_')
    )

# ================== 1. Load shared data (once) ==================
coal_gem = pd.read_excel(COAL_PATH, sheet_name='Units')
country_list = pd.read_excel(COAL_PATH, sheet_name='country')
capacity_factor = pd.read_excel(CF_CAP_PATH)
cf_trend_df = pd.read_csv(CF_PATH)

# ================== 2. Loop parameters ==================
LIFETIMES = [25, 36, 40, 50]
CF_SCENARIOS = ['Limit to 2°C', 'Limit to 3°C', 'Exceed 3°C']
SCENARIO_LABELS = {
    'Limit to 2°C': '<2°C scenario',
    'Limit to 3°C': '2–3°C scenario',
    'Exceed 3°C': '>3°C scenario'
}

# ================== 3. Main loop ==================
summary_rows = []

for lifetime in LIFETIMES:
    sheet_name = f'scenario_{lifetime}'
    life_extend = pd.read_excel(EXCEL_PATH, sheet_name=sheet_name)

    matched = coal_gem.merge(
        life_extend,
        left_on='GEM unit/phase ID',
        right_on='GEM_unit_phase_ID',
        how='inner'
    ).drop(columns=['GEM unit/phase ID'])

    matched = matched.merge(
        country_list[['Country/Area', 'Region_CF', 'Alpha-3 code']],
        on='Country/Area',
        how='left'
    )
    matched = matched.merge(
        capacity_factor[['ISO3', 'CF']],
        left_on='Alpha-3 code',
        right_on='ISO3',
        how='left',
        suffixes=('', '_iso')
    )
    capacity_region = capacity_factor[
        capacity_factor['ISO3'].isna()
    ][['Country', 'CF']].drop_duplicates()
    matched = matched.merge(
        capacity_region,
        left_on='Region_CF',
        right_on='Country',
        how='left',
        suffixes=('', '_region')
    )
    matched['CF'] = matched['CF'].fillna(matched['CF_region'])
    matched = matched.drop(columns=['ISO3', 'Country', 'CF_region'], errors='ignore')

    matched = matched[[
        'GEM_unit_phase_ID', 'Start year', 'time1', 'life_expect',
        'life_extension', 'life_extension_ci_bottom',
        'life_extension_ci_top', 'Capacity (MW)',
        'Heat rate (Btu per kWh)', 'Emission factor (kg of CO2 per TJ)', 'CF'
    ]]

    for cf_col in CF_SCENARIOS:
        scenario_label = SCENARIO_LABELS[cf_col]
        print(f"\n=== lifetime={lifetime}  CF scenario={scenario_label} ===")

        trend_years = cf_trend_df['Year'].values.astype(float)
        trend_cf = cf_trend_df[cf_col].values.astype(float)

        cs_raw = CubicSpline(trend_years, trend_cf, extrapolate=False)
        cf_at_2024 = float(cs_raw(2024))
        cf_at_2100 = float(cs_raw(2100))
        mult_2100 = cf_at_2100 / cf_at_2024

        def cf_multiplier(year):
            year = np.atleast_1d(np.asarray(year, dtype=float))
            result = np.empty_like(year)
            mask_before = year <= trend_years[0]
            mask_after = year >= trend_years[-1]
            mask_mid = ~mask_before & ~mask_after
            result[mask_before] = float(cs_raw(trend_years[0])) / cf_at_2024
            result[mask_after] = mult_2100
            result[mask_mid] = cs_raw(year[mask_mid]) / cf_at_2024
            return float(result[0]) if result.size == 1 else result

        start_year = 2024
        hours_per_year = 365 * 24

        df_final = matched.copy()
        df_final['time1'] = df_final['time1'].fillna(0).astype(int)
        df_final['life_expect'] = df_final['life_expect'].fillna(0).astype(int)
        for col in [
            'life_extension', 'life_extension_ci_bottom',
            'life_extension_ci_top', 'CF', 'Capacity (MW)',
            'Heat rate (Btu per kWh)', 'Emission factor (kg of CO2 per TJ)'
        ]:
            df_final[col] = df_final[col].fillna(0)

        df_final['baseline_years'] = (
            df_final['life_expect'] - df_final['time1'] + 1
        ).clip(lower=0).astype(int)

        end_year = int(
            start_year
            + (df_final['baseline_years'] + np.ceil(df_final['life_extension_ci_top'])).max()
        )

        def annual_co2_for_unit(row, cal_year):
            cf_year = row['CF'] * cf_multiplier(cal_year)
            return (
                hours_per_year * cf_year
                * row['Capacity (MW)'] * 1000
                * row['Heat rate (Btu per kWh)']
                * row['Emission factor (kg of CO2 per TJ)']
                / 9.478e8 / 1000
            )

        def expand_stage_emissions(row, start_year, baseline_years, life_extension, scenario='Central'):
            records = []
            baseline_years = int(max(0, baseline_years))
            life_extension = max(0, life_extension)
            if row['Capacity (MW)'] <= 0 or row['Heat rate (Btu per kWh)'] <= 0:
                return records
            if scenario == 'Central':
                for i in range(baseline_years):
                    records.append({
                        'Year': start_year + i,
                        'Stage': 'Baseline emissions',
                        'Scenario': scenario,
                        'Annual_CO2': annual_co2_for_unit(row, start_year + i)
                    })
            ext_start = start_year + baseline_years
            ext_full = int(np.floor(life_extension))
            ext_rem = life_extension - ext_full
            for i in range(ext_full):
                records.append({
                    'Year': ext_start + i,
                    'Stage': 'Additional emissions',
                    'Scenario': scenario,
                    'Annual_CO2': annual_co2_for_unit(row, ext_start + i)
                })
            if ext_rem > 0:
                cal_year = ext_start + ext_full
                records.append({
                    'Year': cal_year,
                    'Stage': 'Additional emissions',
                    'Scenario': scenario,
                    'Annual_CO2': annual_co2_for_unit(row, cal_year) * ext_rem
                })
            return records

        records = []
        for _, row in df_final.iterrows():
            records += expand_stage_emissions(
                row, start_year, row['baseline_years'], row['life_extension'], scenario='Central'
            )
            records += expand_stage_emissions(
                row, start_year, row['baseline_years'], row['life_extension_ci_bottom'], scenario='Lower'
            )
            records += expand_stage_emissions(
                row, start_year, row['baseline_years'], row['life_extension_ci_top'], scenario='Upper'
            )

        df_yearly = pd.DataFrame(records)

        emissions_central = (
            df_yearly[df_yearly['Scenario'] == 'Central']
            .groupby(['Year', 'Stage'], as_index=False)['Annual_CO2'].sum()
            .pivot(index='Year', columns='Stage', values='Annual_CO2')
            .fillna(0) / 1e6
        )
        emissions_central = emissions_central.reindex(range(start_year, end_year + 1), fill_value=0)
        for col in ['Baseline emissions', 'Additional emissions']:
            if col not in emissions_central.columns:
                emissions_central[col] = 0
        emissions_central = emissions_central[['Baseline emissions', 'Additional emissions']]

        emissions_unc = (
            df_yearly[df_yearly['Stage'] == 'Additional emissions']
            .groupby(['Year', 'Scenario'], as_index=False)['Annual_CO2'].sum()
            .pivot(index='Year', columns='Scenario', values='Annual_CO2')
            .fillna(0) / 1e6
        )
        emissions_unc = emissions_unc.reindex(range(start_year, end_year + 1), fill_value=0)
        for col in ['Lower', 'Upper']:
            if col not in emissions_unc.columns:
                emissions_unc[col] = 0

        baseline_total = emissions_central['Baseline emissions'].sum()
        additional_total = emissions_central['Additional emissions'].sum()
        print(f"  Total baseline emissions:   {baseline_total:.2f} Mt CO2")
        print(f"  Total additional emissions: {additional_total:.2f} Mt CO2")
        summary_rows.append({
            'Lifetime': lifetime,
            'CF_scenario': scenario_label,
            'Baseline_total_Mt': round(baseline_total, 2),
            'Additional_total_Mt': round(additional_total, 2)
        })

        cf_col_safe = scenario_file_label(cf_col)
        out_csv = OUTPUT_TABLE_DIR / f'emissions_life{lifetime}_{cf_col_safe}.csv'
        result_df = emissions_central.copy()
        result_df['Lower'] = emissions_unc['Lower']
        result_df['Upper'] = emissions_unc['Upper']
        result_df.to_csv(out_csv)
        print(f"  Saved: {out_csv}")

        fig, ax = plt.subplots(figsize=(5, 5))
        baseline_area = ax.fill_between(
            emissions_central.index,
            0,
            emissions_central['Baseline emissions'],
            color='#15173D',
            alpha=0.9,
            linewidth=0,
            label='Baseline emissions'
        )
        additional_area = ax.fill_between(
            emissions_central.index,
            emissions_central['Baseline emissions'],
            emissions_central['Baseline emissions'] + emissions_central['Additional emissions'],
            color='#B153D7',
            alpha=0.9,
            linewidth=0,
            label='Additional emissions'
        )
        upper_line, = ax.plot(
            emissions_unc.index,
            emissions_central['Baseline emissions'] + emissions_unc['Upper'],
            color='#7B2CBF',
            linewidth=2.5,
            linestyle='--',
            label='Uncertainty'
        )
        ax.plot(
            emissions_unc.index,
            emissions_central['Baseline emissions'] + emissions_unc['Lower'],
            color='#7B2CBF',
            linewidth=2.5,
            linestyle='--'
        )
        ax.set_xlim(start_year, 2130)
        ax.set_ylim(0, 1500)
        ax.set_yticks([0, 500, 1000, 1500])
        ax.set_ylabel('CO$_2$ emission (Mt)', fontsize=14)
        ax.tick_params(labelsize=12)
        xticks = np.unique(np.append(ax.get_xticks(), start_year))
        xticks = xticks[(xticks >= start_year) & (xticks <= end_year)]
        ax.set_xticks(xticks)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.text(
            0.97,
            0.06,
            f'{scenario_label}\nLifetime={lifetime}yr',
            transform=ax.transAxes,
            fontsize=14,
            ha='right',
            va='bottom'
        )
        ax.legend(
            handles=[baseline_area, additional_area, upper_line],
            labels=['Baseline emissions', 'Additional emissions', 'Uncertainty'],
            fontsize=14,
            loc='upper right',
            frameon=False
        )
        plt.tight_layout()
        out_fig = OUTPUT_FIGURE_DIR / f'plot_life{lifetime}_{cf_col_safe}.svg'
        plt.savefig(out_fig, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {out_fig}")

# ================== 4. Summary table ==================
summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_TABLE_DIR / 'summary_all_scenarios.csv'
summary_df.to_csv(summary_path, index=False)
print(f"\nAll done. Summary saved to: {summary_path}")
print(summary_df.to_string(index=False))


In [ ]:
LIFETIMES = [25, 36, 40, 50]
CF_SCENARIOS = ['Limit to 2°C', 'Limit to 3°C', 'Exceed 3°C']

summary_df = pd.read_csv(TABLES / 'summary_all_scenarios.csv')

fig, axes = plt.subplots(
    len(LIFETIMES),
    len(CF_SCENARIOS),
    figsize=(7.2, 8.8),
    sharex=True,
    sharey=True
)

for row_idx, lifetime in enumerate(LIFETIMES):
    for col_idx, cf_col in enumerate(CF_SCENARIOS):
        ax = axes[row_idx, col_idx]

        cf_col_safe = scenario_file_label(cf_col)
        csv_path = TABLES / f'emissions_life{lifetime}_{cf_col_safe}.csv'
        df = pd.read_csv(csv_path, index_col=0)
        df.index = df.index.astype(int)

        row = summary_df[
            (summary_df['Lifetime'] == lifetime) &
            (summary_df['CF_scenario'] == SCENARIO_LABELS[cf_col])
        ]
        additional_total = row['Additional_total_Mt'].values[0]

        emissions_central = df[['Baseline emissions', 'Additional emissions']]
        emissions_unc = df[['Lower', 'Upper']]

        total_baseline = emissions_central['Baseline emissions']
        total_additional = (
            emissions_central['Baseline emissions'] +
            emissions_central['Additional emissions']
        )

        ax.fill_between(
            emissions_central.index,
            0,
            total_baseline,
            color='#15173D',
            alpha=0.90,
            linewidth=0,
            zorder=1
        )

        ax.fill_between(
            emissions_central.index,
            total_baseline,
            total_additional,
            color='#B153D7',
            alpha=0.88,
            linewidth=0,
            zorder=2
        )

        ax.plot(
            emissions_unc.index,
            total_baseline + emissions_unc['Upper'],
            color='#7B2CBF',
            linewidth=1.25,
            linestyle='--',
            zorder=3
        )

        ax.plot(
            emissions_unc.index,
            total_baseline + emissions_unc['Lower'],
            color='#7B2CBF',
            linewidth=1.25,
            linestyle='--',
            zorder=3
        )

        ax.set_xlim(2024, 2130)
        ax.set_ylim(0, 1500)
        ax.set_yticks([0, 500, 1000, 1500])
        ax.set_xticks([2025, 2050, 2075, 2100, 2125])

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.7)
        ax.spines['bottom'].set_linewidth(0.7)

        ax.tick_params(
            axis='both',
            direction='out',
            labelsize=8,
            width=0.7,
            length=3
        )

        if row_idx == 0:
            ax.set_title(
                SCENARIO_LABELS[cf_col],
                fontsize=10,
                fontweight='bold',
                pad=8
            )

        if col_idx == 0:
            ax.set_ylabel(
                f'{lifetime} yr\n\nCO$_2$ emissions (Mt)',
                fontsize=9
            )
        else:
            ax.tick_params(labelleft=False)

        if row_idx == len(LIFETIMES) - 1:
            ax.set_xlabel('Year', fontsize=9)
        else:
            ax.tick_params(labelbottom=False)

        ax.text(
            0.96,
            0.08,
            f'+{additional_total:.0f} Mt',
            transform=ax.transAxes,
            fontsize=8,
            ha='right',
            va='bottom',
            color='black'
        )

handles = [
    plt.Rectangle((0, 0), 1, 1, fc='#15173D', alpha=0.90),
    plt.Rectangle((0, 0), 1, 1, fc='#B153D7', alpha=0.88),
    plt.Line2D([0], [0], color='#7B2CBF', linewidth=1.25, linestyle='--')
]

labels = [
    'Baseline emissions',
    'Additional emissions',
    'Uncertainty range'
]

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.025),
    ncol=3,
    fontsize=9,
    frameon=False
)

fig.text(
    0.015,
    0.52,
    'Plant lifetime scenario',
    rotation=90,
    va='center',
    ha='center',
    fontsize=10
)

plt.tight_layout(rect=[0.045, 0.05, 1, 0.97], h_pad=0.8, w_pad=0.35)

fig_path = FIGURES / 'Scenarios_12_compact.svg'
plt.savefig(fig_path, bbox_inches='tight')

print(f"Saved: {fig_path}")

### Baseline scenario: 40-year lifetime with the <2°C capacity-factor pathway

In [ ]:
# ---- Single-scenario plot: Fig. 5a-consistent style ----
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

FONT_FAMILY = "Arial"
FONT_SIZE_BASE = 11
FONT_SIZE_AXIS_TEXT = 10
FONT_SIZE_AXIS_TITLE = 11
FONT_SIZE_LEGEND = 10

# ggplot linewidth = 0.8 is approximately 2.25 pt in Matplotlib.
LINE_WIDTH_AXIS = 2.25
LINE_WIDTH_UNCERTAINTY = 2.5

mpl.rcParams["font.family"] = FONT_FAMILY
mpl.rcParams["font.size"] = FONT_SIZE_BASE
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

lifetime = 40
cf_col = "Limit to 2°C"
cf_col_safe = scenario_file_label(cf_col)

csv_path = TABLES / f"emissions_life{lifetime}_{cf_col_safe}.csv"
df = pd.read_csv(csv_path, index_col=0)
df.index = df.index.astype(int)

fig, ax = plt.subplots(figsize=(6, 5))

baseline_area = ax.fill_between(
    df.index,
    0,
    df["Baseline emissions"],
    color="#15173D",
    alpha=0.9,
    linewidth=0,
    label="Baseline emissions",
    clip_on=True
)

additional_area = ax.fill_between(
    df.index,
    df["Baseline emissions"],
    df["Baseline emissions"] + df["Additional emissions"],
    color="#B153D7",
    alpha=0.9,
    linewidth=0,
    label="Additional emissions",
    clip_on=True
)

upper_line, = ax.plot(
    df.index,
    df["Baseline emissions"] + df["Upper"],
    color="#7B2CBF",
    linewidth=LINE_WIDTH_UNCERTAINTY,
    linestyle="--",
    label="Uncertainty",
    clip_on=True
)

ax.plot(
    df.index,
    df["Baseline emissions"] + df["Lower"],
    color="#7B2CBF",
    linewidth=LINE_WIDTH_UNCERTAINTY,
    linestyle="--",
    clip_on=True
)

ax.set_xlim(df.index.min(), df.index.max())

xticks = [2025, 2050, 2075, 2100, 2125]
xticks = [
    x for x in xticks
    if df.index.min() <= x <= df.index.max()
]
ax.set_xticks(xticks)

ax.set_ylim(0, 1500)
ax.set_yticks([0, 500, 1000, 1500])

ax.set_ylabel(
    "CO$_2$ emission (Mt)",
    fontsize=FONT_SIZE_AXIS_TITLE,
    labelpad=8
)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=FONT_SIZE_AXIS_TEXT,
    direction="out",
    length=2,
    width=LINE_WIDTH_AXIS,
    pad=4
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(LINE_WIDTH_AXIS)
ax.spines["bottom"].set_linewidth(LINE_WIDTH_AXIS)

ax.legend(
    handles=[baseline_area, additional_area, upper_line],
    labels=["Baseline emissions", "Additional emissions", "Uncertainty"],
    fontsize=FONT_SIZE_LEGEND,
    loc="upper right",
    frameon=False,
    handlelength=1.8,
    handletextpad=0.5,
    borderaxespad=0.5
)

plt.tight_layout()

fig_path = FIGURES / f"Scenario_single_life{lifetime}_{cf_col_safe}.svg"
plt.savefig(
    fig_path,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none"
)

plt.show()

print(f"Saved: {fig_path}")

In [ ]:
annual_additional = (
    df.loc[df.index >= 2025, ['Additional emissions']]
    .copy()
    .reset_index()
)

annual_additional.columns = [
    'Year',
    'Additional emissions (Mt CO2)'
]

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None
):
    display(annual_additional)

peak_row = annual_additional.loc[
    annual_additional['Additional emissions (Mt CO2)'].idxmax()
]

print(
    f"Peak additional emissions: "
    f"{peak_row['Additional emissions (Mt CO2)']:.2f} Mt CO2 "
    f"in {int(peak_row['Year'])}."
)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

lifetimes = [25, 36, 40, 50]
cf_scenarios = ['Limit to 2°C', 'Limit to 3°C', 'Exceed 3°C']

results = []

for life in lifetimes:
    for scenario in cf_scenarios:
        scenario_safe = scenario_file_label(scenario)
        file_path = TABLES / f'emissions_life{life}_{scenario_safe}.csv'
        df = pd.read_csv(file_path)

        additional_total = df['Additional emissions'].sum()
        lower_total = df['Lower'].sum()
        upper_total = df['Upper'].sum()

        lower_error = additional_total - lower_total
        upper_error = upper_total - additional_total

        results.append({
            'Lifetime': life,
            'Scenario': scenario,
            'Additional': additional_total,
            'Err_lower': lower_error,
            'Err_upper': upper_error
        })

plot_df = pd.DataFrame(results)

scenario_colors = {
    'Limit to 2°C': '#8F0177',
    'Limit to 3°C': '#DE1A58',
    'Exceed 3°C': '#F67D31'
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7.5,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 3,
    'ytick.major.size': 3
})

fig, ax = plt.subplots(figsize=(4.7, 3.0))

x = np.arange(len(lifetimes))
width = 0.20
offsets = [-width, 0, width]

bar_containers = []

for i, scenario in enumerate(cf_scenarios):
    sub = (
        plot_df[plot_df['Scenario'] == scenario]
        .sort_values('Lifetime')
        .copy()
    )

    bars = ax.bar(
        x + offsets[i],
        sub['Additional'].values,
        width=width,
        color=scenario_colors[scenario],
        edgecolor='none',
        yerr=np.vstack([
            sub['Err_lower'].values,
            sub['Err_upper'].values
        ]),
        error_kw={
            'ecolor': 'black',
            'elinewidth': 0.75,
            'capthick': 0.75,
            'capsize': 2.5
        },
        zorder=2
    )
    bar_containers.append(bars)

ax.set_xticks(x)
ax.set_xticklabels([str(l) for l in lifetimes])

ax.set_xlabel('Plant lifetime scenario (years)')
ax.set_ylabel('Additional emissions (Mt CO$_2$)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(direction='out')

ax.grid(False)

scenario_handles = [
    Patch(facecolor=scenario_colors[s], edgecolor='none', label=SCENARIO_LABELS[s])
    for s in cf_scenarios
]

error_handle = Line2D(
    [0],
    [0],
    color='black',
    linewidth=0.75,
    marker='_',
    markersize=6,
    label='Uncertainty range'
)

ax.legend(
    handles=scenario_handles + [error_handle],
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    handlelength=1.4,
    labelspacing=0.55,
    borderaxespad=0
)

fig.tight_layout(pad=0.8)

fig_path_svg = FIGURES / 'Scenarios_summary_nature_style.svg'

fig.savefig(fig_path_svg, bbox_inches='tight')

print(f"Saved: {fig_path_svg}")

plt.show()

In [ ]:
# Display central estimates and uncertainty bounds
summary_display = (
    plot_df
    .assign(
        Lower=lambda x: x['Additional'] - x['Err_lower'],
        Upper=lambda x: x['Additional'] + x['Err_upper']
    )
    [[
        'Lifetime',
        'Scenario',
        'Additional',
        'Lower',
        'Upper'
    ]]
    .sort_values(['Lifetime', 'Scenario'])
    .rename(columns={
        'Lifetime': 'Plant lifetime (years)',
        'Scenario': 'Capacity-factor scenario',
        'Additional': 'Central estimate (Mt CO2)',
        'Lower': 'Lower bound (Mt CO2)',
        'Upper': 'Upper bound (Mt CO2)'
    })
    .reset_index(drop=True)
)

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.float_format', '{:,.2f}'.format
):
    display(summary_display)

In [ ]:
matched